# NayePankh Foundation NGO - Exploratory Data Analysis (EDA)
This Jupyter Notebook performs a detailed exploratory data analysis on the cleaned operational dataset of the NayePankh Foundation to extract insights on program impact, financial efficiency, donor distributions, and volunteer leverage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load cleaned data
df = pd.read_csv('../data/cleaned_ngo_data.csv')
df['Enrollment Date'] = pd.to_datetime(df['Enrollment Date'])
df.head()

### 1. Dataset Shape and Column Overview

In [ ]:
print(f"Dataset Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.info()

### 2. High-Level Summary Statistics

In [ ]:
df.describe().T

### 3. Program Enrollment and Completion Rates by Category

In [ ]:
# Calculate enrollments and completion rates
prog_stats = df.groupby('Program Category').agg(
    Total_Enrollments=('Beneficiary ID', 'count'),
    Avg_Attendance=('Attendance Percentage', 'mean'),
    Completion_Rate=('Completion Success', 'mean'),
    Avg_Satisfaction=('Satisfaction Score', 'mean')
).reset_index()

prog_stats['Completion_Rate'] = (prog_stats['Completion_Rate'] * 100).round(2)
prog_stats['Avg_Attendance'] = prog_stats['Avg_Attendance'].round(2)
prog_stats['Avg_Satisfaction'] = prog_stats['Avg_Satisfaction'].round(2)
prog_stats

### 4. Visualizing Financial Efficiency (Allocation vs Utilization)

In [ ]:
fund_summary = df.groupby('Program Category')[['Funds Allocated', 'Funds Utilized']].sum().reset_index()
fund_melted = pd.melt(fund_summary, id_vars='Program Category', value_vars=['Funds Allocated', 'Funds Utilized'],
                      var_name='Budget Type', value_name='Amount (INR)')

sns.barplot(data=fund_melted, x='Program Category', y='Amount (INR)', hue='Budget Type', palette=['#1E88E5', '#FF5722'])
plt.title('Funds Allocated vs Utilized by Program Category')
plt.xticks(rotation=15)
plt.ylabel('Amount in Millions (INR)')
plt.show()

### 5. Employment Success Rates (Skill Development & Digital Literacy)

In [ ]:
jobs_df = df[df['Program Category'].isin(['Skill Development', 'Digital Literacy']) & (df['Program Completion Status'] == 'Completed')]
emp_rates = jobs_df.groupby('Program Category')['Employment Obtained After Training (Yes/No)'].value_counts(normalize=True).unstack() * 100
emp_rates = emp_rates.round(2)
print("Employment success percentages for graduated students:")
emp_rates

### 6. Correlation Analysis

In [ ]:
numeric_cols = ['Age', 'Attendance Percentage', 'Training Hours', 'Funds Allocated', 'Funds Utilized', 'Volunteer Count', 'Satisfaction Score', 'Event Participation Count']
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Quantitative NGO Metrics')
plt.show()